In [ ]:
import pandas as pd
from utils import *
import split_lookups
import datetime
import shutil
import os

In [ ]:
script_dir = os.path.dirname(os.path.abspath(__file__))
split_lookups = os.path.join(script_dir, 'Split_Lookup_Lists')
#parameterize the split_lookup file name?

input_file = os.path.join(script_dir,'download_variables.csv')

split_csv(input_file, split_lookups)

In [ ]:
# Run split_lookups

In [ ]:
%run 'Download_Wrapper.py'

In [ ]:
#Import all csvs from a folder and merge them into a single dataframe
output_folder = r'C:\Users\amcclary\Documents\GitHub\Housing\Scripts\Dowloaded_Data'
def import_csvs(folder_path):
    import os
    import glob
    os.chdir(folder_path)
    extension = 'csv'
    all_filenames = [i for i in glob.glob('*.{}'.format(extension))]
    #combine all files in the list
    combined_csv = pd.concat([pd.read_csv(f) for f in all_filenames ])
    #move all of the csv files to a new folder
    folder_path_appended = os.path.join(folder_path, 'appended_csvs')
    os.makedirs(folder_path_appended, exist_ok=True)
    for f in all_filenames:
        shutil.move(f, 'appended_csvs')
    return combined_csv
combined_csv = import_csvs(output_folder)

#add a leading 0 to the TRPAID column if the length is 15
def add_leading_zero_based_on_condition(row):
    if row['sample_level'] == 'tract' and len(row['TRPAID']) == 14:
        return row['TRPAID'].zfill(15)
    elif row['sample_level'] == 'block group' and len(row['TRPAID']) == 15:
        return row['TRPAID'].zfill(16)
    else:
        return row['TRPAID']

combined_csv['TRPAID'] = combined_csv['TRPAID'].astype(str)
combined_csv['TRPAID'] = combined_csv.apply(add_leading_zero_based_on_condition, axis=1)
combined_csv
combined_csv['state'] = combined_csv['state'].astype(str).str.zfill(2)
combined_csv['county'] = combined_csv['county'].astype(str).str.zfill(3)
# Drop rows with a missing value
combined_csv = combined_csv.dropna(subset=['value'])


In [ ]:
# Map pandas dtypes to ArcGIS field types - need to do this or weird things happen
combined_csv['GEO_ID'] = combined_csv['GEO_ID'].astype(str)
type_mapping = {
    'int64': 'LONG',
    'float64': 'DOUBLE',
    'object': 'TEXT',
    'string': 'TEXT',
    'datetime64[ns]': 'DATE'
}


# Set up geodatabase and output table name
gdb_path = r"F:\GIS\PROJECTS\ResearchAnalysis\Demographics\Workspace.gdb"
#add a data stamp to the table name

now = datetime.datetime.now()
data_stamp = now.strftime("%Y%m%d")
output_table = "census_data_append" + data_stamp
output_path = f"{gdb_path}\\{output_table}"
if arcpy.Exists(output_path):
    arcpy.management.Delete(output_path)
    print(f"Deleted existing table: {output_table}")
# Create the table in the geodatabase
arcpy.management.CreateTable(gdb_path, output_table)

# Add fields based on DataFrame dtypes
for col_name, dtype in combined_csv.dtypes.items():
    arcgis_type = type_mapping.get(str(dtype), 'TEXT')  # Default to TEXT if dtype is unknown
    if arcgis_type == 'TEXT':
        arcpy.management.AddField(output_path, col_name, arcgis_type, field_length=255)
    else:
        arcpy.management.AddField(output_path, col_name, arcgis_type)

# Insert data into the table
with arcpy.da.InsertCursor(output_path, combined_csv.columns.tolist()) as cursor:
    for _, row in combined_csv.iterrows():
        cursor.insertRow(row.tolist())

print(f"Table '{output_table}' created and populated in {gdb_path}")

In [ ]:
# we could auto append but for now I want to keep review on the data